In [ ]:
import csv
import os
import re
import time
import random
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Union, Tuple

import requests
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\2nd_Try")

TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

URL_LIST_CSV = ROOT / "URL_List_Instru.csv"
EPISODES_CSV  = ROOT / "List_Change_Episodes.csv"

OUT_DIR = ROOT / "RouteA_EpisodeRunMetrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Raw output (incremental write)
RAW_OUT = OUT_DIR / "routeA_runs_labeled.csv"

# Stop condition: collect K instrumentation-related CI records per episode
K_INSTRU_RECORDS_PER_EPISODE = 100

# Time-window growth
INITIAL_WINDOW_DAYS = 30
MAX_WINDOW_DAYS = 3650  # 10 years cap (safety)

# Safety cap
MAX_COMMITS_TO_SCAN_PER_EPISODE = 5000

# Provider focus:
#   "gha_only"     -> only GitHub Actions check-runs (skip non-GHA check-runs + skip commit statuses)
#   "non_gha_only" -> only non-GHA check-runs + commit statuses (skip GHA check-runs)
#   "both"         -> keep everything
PROVIDER_FOCUS = "both"   # <- change to "gha_only" or "non_gha_only" when needed

# Retry/backoff
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# ============================================================
# Instrumentation-ish matching
# ============================================================
INSTRU_NAME_RE = re.compile(
    r"(androidtest|connectedandroidtest|connected.*android.*test|connectedcheck|devicecheck|"
    r"manageddevice|gmd|instrument(ed|ation)?|am\s+instrument|espresso|uiautomator|ui\s*test|"
    r"firebase\s+test|test\s*lab|device\s*farm|browserstack|sauce(labs)?|kobiton|appcenter|"
    r"flank|maestro|marathon|spoon|baselineprofile|benchmark)",
    re.IGNORECASE,
)

# ============================================================
# Helpers
# ============================================================
def flush_rows(rows: List[Dict], out_path: Path):
    """Append rows to CSV immediately (episode by episode)."""
    if not rows:
        return
    write_header = not out_path.exists()
    with out_path.open("a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        if write_header:
            w.writeheader()
        w.writerows(rows)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def parse_repo_full_name(repo_url_or_fullname: str) -> Optional[str]:
    s = (repo_url_or_fullname or "").strip()
    if not s:
        return None
    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", s):
        return s
    m = re.search(r"github\.com[:/]+([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", s, re.IGNORECASE)
    if m:
        full = m.group(1)
        return full[:-4] if full.endswith(".git") else full
    return None

def allow_check_run(app_slug: str) -> bool:
    slug = (app_slug or "").strip().lower()
    if PROVIDER_FOCUS == "both":
        return True
    if PROVIDER_FOCUS == "gha_only":
        return slug == "github-actions"
    if PROVIDER_FOCUS == "non_gha_only":
        return slug != "github-actions"
    return True

def allow_commit_status() -> bool:
    # Commit statuses are typically for non-GHA providers.
    if PROVIDER_FOCUS == "gha_only":
        return False
    return True

def is_instru(name_or_context: str) -> bool:
    return bool(INSTRU_NAME_RE.search(name_or_context or ""))

def seconds_between(a: Optional[str], b: Optional[str]) -> Optional[int]:
    try:
        da = pd.to_datetime(a, utc=True)
        db = pd.to_datetime(b, utc=True)
        sec = int((db - da).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def load_tokens_from_env_file(env_path: Path) -> List[str]:
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}")
    return tokens

def read_repo_list_from_url_csv(path: Path) -> List[str]:
    df = pd.read_csv(path, encoding="utf-8-sig")
    cols_l = {c.lower(): c for c in df.columns}

    cand_cols = []
    for k in ["html_url", "url", "repo_url", "repository", "repo", "full_name"]:
        if k in cols_l:
            cand_cols.append(cols_l[k])

    if not cand_cols:
        cand_cols = [df.columns[0]]

    out = []
    for col in cand_cols[:1]:
        for v in df[col].astype(str).tolist():
            full = parse_repo_full_name(v)
            if full:
                out.append(full)

    return sorted(set(out))

def build_repo_canon_map(repos: List[str]) -> Dict[str, str]:
    return {r.lower(): r for r in repos}

def normalize_episode_repo(row: pd.Series, repo_map: Dict[str, str]) -> Optional[str]:
    candidates: List[str] = []
    for key in ["repo_name", "full_name", "html_url", "url", "repo", "repository", "repo_url"]:
        if key in row and pd.notna(row[key]):
            s = str(row[key]).strip()
            if s:
                candidates.append(s)

    for raw in candidates:
        p = parse_repo_full_name(raw)
        if p and p.lower() in repo_map:
            return repo_map[p.lower()]

        if "__" in raw and raw.count("__") == 1:
            p2 = raw.replace("__", "/")
            if p2.lower() in repo_map:
                return repo_map[p2.lower()]

        if "/" not in raw and "." in raw:
            p3 = raw.replace(".", "/", 1)
            if p3.lower() in repo_map:
                return repo_map[p3.lower()]

    return None

# ============================================================
# GitHub client
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "episode-ci-metrics-k/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        time.sleep(max(1, soonest - now + 2))

    def _backoff(self, attempt: int) -> None:
        time.sleep(min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random())

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code
            try:
                st.remaining = int(resp.headers.get("X-RateLimit-Remaining", "0"))
            except Exception:
                pass
            try:
                st.reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
            except Exception:
                pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and ("rate limit" in text_l or "secondary rate limit" in text_l):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate_list(self, url: str, params: Optional[Dict] = None) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if not data or not isinstance(data, list):
                return
            if not data:
                return
            for it in data:
                yield it
            if len(data) < 100:
                return
            page += 1

# ============================================================
# GitHub endpoints
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    url = f"https://api.github.com/repos/{full_name}"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_commits_in_range(
    gh: GitHubClient,
    full_name: str,
    branch: str,
    since_iso: str,
    until_iso: str,
) -> List[str]:
    url = f"https://api.github.com/repos/{full_name}/commits"
    params = {"sha": branch, "since": since_iso, "until": until_iso}
    shas: List[str] = []
    for c in gh.paginate_list(url, params=params):
        sha = c.get("sha")
        if isinstance(sha, str) and len(sha) >= 7:
            shas.append(sha)
    return shas  # newest -> oldest

def list_check_runs_for_commit(gh: GitHubClient, full_name: str, sha: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/check-runs"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return []
    return data.get("check_runs", []) or []

def get_combined_status_for_commit(gh: GitHubClient, full_name: str, sha: str) -> Dict:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/status"
    data = gh.request_json("GET", url)
    return data if isinstance(data, dict) else {}

# ============================================================
# Episodes loader
# ============================================================
def load_episodes(path: Path, repo_map: Dict[str, str]) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    if "episode_start_utc" not in df.columns or "episode_end_utc" not in df.columns:
        raise ValueError(f"Missing episode_start_utc/episode_end_utc in {path.name}. Columns={list(df.columns)}")

    out = df.copy()
    out["episode_start_utc"] = pd.to_datetime(out["episode_start_utc"], utc=True, errors="coerce")
    out["episode_end_utc"]   = pd.to_datetime(out["episode_end_utc"], utc=True, errors="coerce")

    style_col = "env_styles" if "env_styles" in out.columns else None
    out["episode_env_styles"] = out[style_col].astype(str) if style_col else ""

    if "episode_id" in out.columns:
        out["episode_id"] = out["episode_id"]
    elif "episode_index" in out.columns:
        out["episode_id"] = out["episode_index"]
    else:
        out["episode_id"] = range(len(out))

    out["full_name"] = out.apply(lambda r: normalize_episode_repo(r, repo_map), axis=1)
    out = out.dropna(subset=["full_name", "episode_start_utc", "episode_end_utc"])
    return out[["full_name", "episode_id", "episode_start_utc", "episode_end_utc", "episode_env_styles"]]

# ============================================================
# Main
# ============================================================
def main():
    if not URL_LIST_CSV.exists():
        raise FileNotFoundError(f"URL list not found: {URL_LIST_CSV}")
    if not EPISODES_CSV.exists():
        raise FileNotFoundError(f"Episodes file not found: {EPISODES_CSV}")
    if not TOKENS_ENV_PATH.exists():
        raise FileNotFoundError(f"Tokens env not found: {TOKENS_ENV_PATH}")

    # If re-running: remove old raw file so header isn't duplicated and file is "fresh"
    if RAW_OUT.exists():
        RAW_OUT.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens)

    repos = read_repo_list_from_url_csv(URL_LIST_CSV)
    repo_map = build_repo_canon_map(repos)
    episodes = load_episodes(EPISODES_CSV, repo_map)

    print(f"[load] repos={len(repos)} (from {URL_LIST_CSV.name})")
    print(f"[load] episodes(total)={len(pd.read_csv(EPISODES_CSV, encoding='utf-8-sig'))}")
    print(f"[load] episodes(matched to repos)={len(episodes)}")

    if len(episodes) == 0:
        print("[ERROR] No episodes matched your URL list.")
        return

    default_branch_cache: Dict[str, str] = {}

    # Group per repo
    for full_name, grp in episodes.groupby("full_name"):
        if full_name not in default_branch_cache:
            default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
        branch = default_branch_cache[full_name]
        if not branch:
            continue

        for _, ep in grp.iterrows():
            start_dt = ep["episode_start_utc"]
            end_dt   = ep["episode_end_utc"]
            if pd.isna(start_dt) or pd.isna(end_dt) or start_dt >= end_dt:
                continue

            collected = 0
            commits_scanned = 0
            seen_shas = set()

            cursor = start_dt
            window_days = INITIAL_WINDOW_DAYS

            raw_rows: List[Dict] = []  # <- episode-local buffer for flushing

            while (
                cursor < end_dt
                and collected < K_INSTRU_RECORDS_PER_EPISODE
                and commits_scanned < MAX_COMMITS_TO_SCAN_PER_EPISODE
            ):
                window_end = min(end_dt, cursor + timedelta(days=window_days))
                since_iso = cursor.isoformat().replace("+00:00", "Z")
                until_iso = window_end.isoformat().replace("+00:00", "Z")

                shas_newest_to_oldest = list_commits_in_range(gh, full_name, branch, since_iso, until_iso)
                if not shas_newest_to_oldest:
                    cursor = window_end
                    window_days = min(MAX_WINDOW_DAYS, max(window_days * 2, window_days + 1))
                    continue

                shas = list(reversed(shas_newest_to_oldest))  # oldest -> newest

                for sha in shas:
                    if collected >= K_INSTRU_RECORDS_PER_EPISODE or commits_scanned >= MAX_COMMITS_TO_SCAN_PER_EPISODE:
                        break
                    if sha in seen_shas:
                        continue
                    seen_shas.add(sha)
                    commits_scanned += 1

                    # ---- Check runs ----
                    for cr in list_check_runs_for_commit(gh, full_name, sha):
                        app = cr.get("app") or {}
                        app_slug = (app.get("slug") or "").strip().lower()

                        if not allow_check_run(app_slug):
                            continue

                        name = (cr.get("name") or "").strip()
                        if not is_instru(name):
                            continue

                        started_at = cr.get("started_at") or ""
                        completed_at = cr.get("completed_at") or ""
                        dur = seconds_between(started_at, completed_at)
                        concl = cr.get("conclusion") or cr.get("status") or ""

                        raw_rows.append({
                            "full_name": full_name,
                            "default_branch": branch,
                            "episode_id": ep["episode_id"],
                            "episode_env_styles": ep["episode_env_styles"],
                            "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                            "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                            "commit_sha": sha,

                            "record_type": "check_run",
                            "provider": app_slug or "unknown_app",
                            "provider_kind": "github_check_run",
                            "job_name": name,
                            "conclusion": concl,
                            "started_at": started_at,
                            "completed_at": completed_at,
                            "duration_seconds": dur if dur is not None else "",

                            "html_url": cr.get("html_url") or "",
                            "collected_at_utc": now_utc_iso(),
                            "run_instance_key": f"checkrun:{cr.get('id')}",
                        })
                        collected += 1
                        if collected >= K_INSTRU_RECORDS_PER_EPISODE:
                            break

                    if collected >= K_INSTRU_RECORDS_PER_EPISODE:
                        break

                    # ---- Commit statuses ----
                    if allow_commit_status():
                        st = get_combined_status_for_commit(gh, full_name, sha)
                        statuses = st.get("statuses", []) if isinstance(st, dict) else []
                        for s in statuses:
                            context = (s.get("context") or "").strip()
                            if not is_instru(context):
                                continue

                            state = (s.get("state") or "").strip().lower()
                            created_at = s.get("created_at") or ""
                            updated_at = s.get("updated_at") or ""
                            dur = seconds_between(created_at, updated_at)

                            prov = (context.split("/")[0] if "/" in context else context.split(":")[0]).strip().lower()

                            raw_rows.append({
                                "full_name": full_name,
                                "default_branch": branch,
                                "episode_id": ep["episode_id"],
                                "episode_env_styles": ep["episode_env_styles"],
                                "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                "commit_sha": sha,

                                "record_type": "commit_status",
                                "provider": prov or "unknown_status",
                                "provider_kind": "github_commit_status",
                                "job_name": context,
                                "conclusion": state,
                                "started_at": "",
                                "completed_at": "",
                                "duration_seconds": dur if dur is not None else "",

                                "html_url": s.get("target_url") or "",
                                "collected_at_utc": now_utc_iso(),
                                "run_instance_key": f"status:{sha}:{context}:{created_at}",
                            })
                            collected += 1
                            if collected >= K_INSTRU_RECORDS_PER_EPISODE:
                                break

                cursor = window_end
                window_days = min(MAX_WINDOW_DAYS, max(window_days * 2, window_days + 1))

            print(f"[episode] {full_name} ep={ep['episode_id']} style={ep['episode_env_styles']} "
                  f"collected={collected} commits_scanned={commits_scanned}")

            # >>> Incremental write HERE (end of episode)
            flush_rows(raw_rows, RAW_OUT)
            raw_rows.clear()

    # ============================================================
    # Aggregations (A) and (B): read from the raw CSV that we wrote incrementally
    # ============================================================
    if not RAW_OUT.exists():
        print("[warn] No raw output file created; nothing to aggregate.")
        return

    df = pd.read_csv(RAW_OUT, encoding="utf-8-sig")
    print(f"[save] {RAW_OUT} rows={len(df)}")

    if len(df) == 0:
        print("[warn] Raw file is empty: no instrumentation-related checks/statuses found.")
        return

    c = df["conclusion"].astype(str).str.lower()
    df["is_success"] = c.isin({"success", "neutral", "skipped"})
    df["is_failure"] = c.isin({"failure", "error", "cancelled", "timed_out", "action_required"})
    df["duration_s"] = pd.to_numeric(df["duration_seconds"], errors="coerce")

    # (A) Repo × Episode(style) × Provider × Job metrics
    gcols = ["full_name", "episode_id", "episode_env_styles", "provider_kind", "provider", "job_name"]
    wf_ep = df.groupby(gcols).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success", "sum"),
        failure_runs=("is_failure", "sum"),
        success_rate=("is_success", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
        dur_p95_s=("duration_s", lambda x: x.dropna().quantile(0.95) if x.dropna().size else float("nan")),
    ).reset_index()

    wf_ep_out = OUT_DIR / "routeA_workflow_episode_metrics.csv"
    wf_ep.to_csv(wf_ep_out, index=False, encoding="utf-8-sig")
    print(f"[save] {wf_ep_out} rows={len(wf_ep)}")

    # (B) Style overall metrics
    style = df.groupby(["episode_env_styles"]).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success", "sum"),
        failure_runs=("is_failure", "sum"),
        success_rate=("is_success", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
    ).reset_index()

    style_out = OUT_DIR / "routeA_style_overall_metrics.csv"
    style.to_csv(style_out, index=False, encoding="utf-8-sig")
    print(f"[save] {style_out} rows={len(style)}")

    print("Done.")

if __name__ == "__main__":
    main()


[load] repos=481 (from URL_List_Instru.csv)
[load] episodes(total)=488
[load] episodes(matched to repos)=487
[episode] 4eRTuk/audioview ep=1 style=Emu_Custom collected=0 commits_scanned=39
[episode] AAkira/ExpandableLayout ep=1 style=Emu_Custom collected=0 commits_scanned=97
[episode] AChep/AcDisplay ep=1 style=Emu_Custom collected=0 commits_scanned=898
[episode] ActivityWatch/aw-android ep=1 style=Emu_Custom collected=0 commits_scanned=75
[episode] AdamMc331/AndroidStudyGuide ep=1 style=Emu_Community collected=0 commits_scanned=18
[episode] AdevintaSpain/Barista ep=1 style=Emu_Custom collected=0 commits_scanned=254
[episode] AdevintaSpain/Barista ep=2 style=Emu_Community collected=0 commits_scanned=96
[episode] Albert221/FastShopping ep=1 style=Emu_Community collected=0 commits_scanned=74
[episode] AlphaWallet/alpha-wallet-android ep=1 style=Emu_Custom collected=0 commits_scanned=5000
[episode] AlphaWallet/alpha-wallet-android ep=2 style=Emu_Community collected=0 commits_scanned=538
[